In [ ]:
# Import various libraries 

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii

# This is used to debayer data - we won't be using it today
# import cv2

import astroalign as aa

from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.detection import DAOStarFinder
from photutils.background import Background2D
from photutils.segmentation  import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import gc                               

from astropy.table import QTable

from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia


# SeeStar

## Question: What is an advantage of using the first SeeStar image for aligning the b, g and r images?

### <font color='blue'>Answer </font>


## Question: Do you need to use the same reference image for scaling and alignment?

### <font color='blue'>Answer </font>


## Calibrating with Gaia Photometry 

Because of the widespread availability of excellent Gaia photometry, we can use it to calibrate photometry (in the Vega system) and to verify photometry. 

For the SeeStar telescopes (or any telescope), I can plot fluxes as a function of photometry to determine relationships between SeeStar photometry and Gaia photometry. To do this well we ideally need a lot of unsaturated main sequence stars with a spread of colours. Older open clusters are ideal.

These equations below are for $G_{BP}-G<1$ or $G-G_{RP}<1$ respectively.

Example calibration

<IMG SRC='seestar_gaia_cal.png' width=400>

$r_{SeeStar} = zp - 2.5\times {\rm log}(f)$

...so...

$+2.5\times {\rm log}(f) = zp - r_{SeeStar}$

So the y axis here is 

$y = G_{RP} - r_{SeeStar} + zp$ 

By definition main-sequence stars with colours of zero are zero in all bands (in a Vega based system) so the y-axis intercept gives us the zero point. Also the plot is effectively measuring the difference between the Gaia and SeeStar photometry as a function of Gaia colour. 

## The following equations are critical for the projects 

SeeStar b-band:

\begin{equation}
b_{SeeStar} = G_{BP} + 0.385 \times (G_{BP}-G)
\end{equation}

SeeStar g-band:

\begin{equation}
g_{SeeStar} = G + 0.155 \times (G_{BP}-G)^2 + 0.443 \times (G_{BP}-G)
\end{equation}

SeeStar r-band:

\begin{equation}
r_{SeeStar} = G_{RP} + 0.370 \times (G-G_{RP})^2 + 0.572 \times (G-G_{RP})
\end{equation}



## Let's get GAIA photometry for an example star

Let's find an example star in a SeeStar default stacked images 


In [ ]:
coord = SkyCoord(ra=101.6459, dec=-20.7128899, unit=(u.degree, u.degree), frame='icrs')
Gaia.ROW_LIMIT = 10
r = Gaia.cone_search_async(coord, radius=u.Quantity(2.0e-3, u.deg)).get_results()
#print(r[0])
print('\n\n')
print('phot_bp_mean_mag: ', r[0]['phot_bp_mean_mag'])
print('phot_g_mean_mag: ', r[0]['phot_g_mean_mag'])
print('phot_rp_mean_mag: ', r[0]['phot_rp_mean_mag'])

# If this query failes then we can use the following numbers:
# ra=101.6459, dec=-20.7128899
# phot_bp_mean_mag:  9.341423
# phot_g_mean_mag:  9.368965
# phot_rp_mean_mag:  9.394363


## Question: What are the corresponding magnitudes in the SeeStar blue, green and red filters?

### <font color='blue'>Answer </font>


## Question: How could I use this single star to determine magnitudes?

### <font color='blue'>Answer </font>

\begin{equation}
m_1 - m_2 = -2.5{\rm log} (f_1/f_2)
\end{equation}


## Question: How could I use this star to define a zeropoint?

### <font color='blue'>Answer </font>


## Question: How would I use multiple stars to define a zeropoint?

### <font color='blue'>Answer </font>


## Background subtraction

When taking the SeeStar data you will have often seen a non-uniform backgroud, which ideally we need to model and subtract before combining.

We can do this with photutils Background2D.

In [ ]:
# Comment
dirname='M_47_sub_debayer/'

# Comment
bimages = ccdproc.ImageFileCollection(dirname,glob_include='*_b.fit')
im_b = [CCDData.read(dirname+filename, unit="adu") for filename in bimages.files_filtered()]
print('b images loaded: ', len(im_b))
      
gimages = ccdproc.ImageFileCollection(dirname,glob_include='*_g.fit')
im_g = [CCDData.read(dirname+filename, unit="adu") for filename in gimages.files_filtered()]
print('g images loaded: ', len(im_g))

rimages = ccdproc.ImageFileCollection(dirname,glob_include='*_r.fit')
im_r = [CCDData.read(dirname+filename, unit="adu") for filename in rimages.files_filtered()]
print('r images loaded: ', len(im_r))


In [ ]:
# Plot a single g-band exposures
vmin=np.nanpercentile(im_g[0], 1)
vmax=np.nanpercentile(im_g[0], 99)

plt.imshow(im_g[0], vmin=vmin, vmax=vmax, origin='lower', cmap='gray')
plt.title('g exposure')
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=1.0)
plt.show()

# Model the data using a 50x50 box size for smoothing 
im_bkg = Background2D(im_g[0], 50)
# Background
print(im_bkg.background)

# Plot the background - Note the scale
vmin=np.nanpercentile(im_bkg.background.value, 1)
vmax=np.nanpercentile(im_bkg.background.value, 99)
plt.imshow(im_bkg.background.value, vmin=vmin, vmax=vmax, origin='lower', cmap='gray')
plt.title('Background')
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=1.0)
plt.show()


# Subtract the background and plot 
testim = im_g[0].data - im_bkg.background.value

# Background subtracted 
vmin=np.nanpercentile(testim, 1)
vmax=np.nanpercentile(testim, 99)
plt.imshow(testim, vmin=vmin, vmax=vmax, origin='lower', cmap='gray')
plt.title('Background subtracted')
plt.xlabel('x')
plt.xlabel('y')
plt.colorbar(shrink=1.0)
plt.show()




# Giradi isochrones

Isochrone - lines of constant age - show how a population of stars looks at specific ages. There's many available in the literature (you have may used some in ASP units already). Isochrones can be used to determine distance, age and dust obsucration (the latter in combination with a dust extinction law).

The Girardi et al. isochrones are available via a web interfacce

http://stev.oapd.inaf.it/cgi-bin/cmd

Many options for the models, but we will primarily use age and metallicity (composition) 

Metallicity (composition) is likely to change as we look at older objects formed when there was less metals in the Universe 


In [ ]:
class isochroneclass:
    def __init__(self):
            self.filename = "NULL                            "      # Relevant filename
            self.age = -99.0                                        # Age
            self.m = [-99.0]*1000                                   # Star masses
            self.U = [-99.0]*1000                                   # Star U-band magnitudes
            self.B = [-99.0]*1000                                   # Star B-band magnitudes
            self.V = [-99.0]*1000                                   # Star V-band magnitudes
            self.R = [-99.0]*1000                                   # Star R-band magnitudes
            self.I = [-99.0]*1000                                   # Star I-band magnitudes

In [ ]:
# Define the input file names - this is the output with the defaults http://stev.oapd.inaf.it/cgi-bin/cmd
# I've renamed the output files so the ages are clear
# Can you comment on what each line is doing?

fnames=['output738046599625_10Myr.txt', 'output495299961724_20Myr.txt', 'output857201723921_40Myr.txt']
isochrone=[]

age=10
for fname in fnames:

    tchrone=isochroneclass()
    tchrone.age=age

    f=open(fname,"r")
    lines=f.readlines()
    idx=0
    for x in lines:
#        if x[0]=='#' and x[2]=='Z':
#            print(x.split()[28])
        if x[0]!='#' and idx<1000:
            tchrone.m[idx]=float(x.split()[5])
            tchrone.U[idx]=float(x.split()[28])
            tchrone.B[idx]=float(x.split()[29])
            tchrone.V[idx]=float(x.split()[30])
            tchrone.R[idx]=float(x.split()[31])
            tchrone.I[idx]=float(x.split()[32])
            idx=idx+1
    f.close()
    isochrone.append(tchrone)
    age=age*2

print('Check the first line from the first file is good')
print(isochrone[0].m[0])
print(isochrone[0].U[0])
print(isochrone[0].B[0])
print(isochrone[0].V[0])
print(isochrone[0].R[0])
print(isochrone[0].I[0])


In [ ]:
# Define plot size
plt.rcParams['figure.figsize'] = [7, 7]

for ic in isochrone:
    plt.axis([-0.5, 2, 10, -10.0])           # Axes ranges
    BV=np.array(ic.B)-np.array(ic.V)
    plt.scatter(BV, ic.V, label=str(ic.age)+' Myr', s=0.5)                 # Scatter plot

# Axis labels and grid
plt.xlabel('B-V colour')          
plt.ylabel('Absolute V-band Magnitude ')
plt.title('Girardi isochrone')
plt.legend(loc='lower left')
plt.grid(True)

# Output file, if wanted
# plt.savefig("test.png")

# Plot to screen
plt.show()

# Pleiades Example 

Photometry from https://webda.physics.muni.cz/cgi-bin/rdb_list_ref.cgi?mel022+ubv.peo+25

I have converted this data to an ASCII file - take a look at it

It has missing values and this makes it a little trickier to read than a CSV file 

I have to specify which characters in each line of text are allocated to which variables

In [ ]:
# Create lists for the parameters
Number=[]
Refer=[]
V=[]
BV=[]
UB=[]
N=[]

# Open the file 
f = open('Pleiades_UBV_photometry.txt', 'r')  # We need to re-open the file
lines = f.readlines()
f.close()

# Go through the file and get the valid data 
for line in lines:
    # print(line)
    if line[0]!='#' and line[15]!=' ':
        # print(line)
        Number.append(line[0:4])
        Refer.append(line[5:10])
        V.append(float(line[10:18]))
        if line[22]!=' ':
            BV.append(float(line[19:26]))
        else:
            BV.append(-99.99)
        if line[30]!=' ':
            UB.append(float(line[27:34]))
        else:
            UB.append(-99.99)
        N.append(line[35:39])


# Create table

There are a number of ways of bundling data into tables in Python

The example below astropy tables 

The lists we created before are put into columns of the table 

It is easier having a table rather than many seperate lists 

In [ ]:
# Create an astropy table

Pleiades = QTable([Number, Refer, V, BV, UB, N], names=('Star number', 'Reference', 'V', 'B-V', 'U-B', 'N Observations'))

print(Pleiades)         # Comment

print(Pleiades[3:8])    # Comment

In [ ]:
px=[]
py=[]
for star in Pleiades:
    if star['V']>0.0 and star['B-V']>-90.0:
        py.append(star['V'])
        px.append(star['B-V'])

plt.gca().invert_yaxis()
plt.scatter(px,py,marker=".")
plt.title('Pleiades HR diagram')
plt.xlabel('$B-V$')
plt.ylabel('$V$')
plt.gca().set_xlim([-0.4, 2.0])


# Adaptive optics 

Our atmosphere blurs our images with a scale on the order of an arcsecond

This results in star images being almost Gaussian blurs of point sources

The blurring is effectively crumpling of wavefronts passing through turbulent layers of the atmosphere

Central to adaptive optics are variations of:

\begin{equation}
{\rm sin} \theta \simeq \lambda / d
\end{equation}

$\theta$ is an angle and $\lambda$ is the wavelength. 

## Question: What are two examples of what d can be?

### <font color='blue'>Answer </font>


The images are broken up into speckles where each isophase patch results in a  diffraction limited of the star (or other celestial object). 

As the isophase patches correspond to turbulent cells they are in rapid motion.

Ideally we would like these speckles to not be moving around but all part of one stable image. 

To do this we need to correct for atmospheric turbulence by measuring the crumpling of wavefronts. 

## Question: How can we measure the tilt of wavefronts? 

### <font color='blue'>Answer </font>

## Question: How can we measure the crumpling of wavefronts? 

### <font color='blue'>Answer </font>



## Question: What are we looking at here? 

<IMG SRC='https://www.ing.iac.es//astronomy/development/hap/figs/anim.gif' width=600>

Taken from https://www.ing.iac.es//astronomy/development/hap/jose.html

### <font color='blue'>Answer </font>
